# Attention: one head, by hand

```
+--------------------------------------------------------------+
|  ONE HEAD OF ATTENTION                                       |
|                                                              |
|  A question arrives. A shelf of possible answers waits.      |
|  Score every answer against the question, turn the scores    |
|  into weights, and blend the answers by those weights.       |
|                                                              |
|  That is the whole mechanism. You will write it in five      |
|  short pieces, then run it inside a real trained model and   |
|  check that it gives the same numbers the model gives.       |
+--------------------------------------------------------------+
```

## What is this?

In the lecture you saw attention twice. First as a shopper choosing between
shops, then as the block diagram: embeddings go in, three matrices turn them
into queries, keys and values, a softmax turns scores into weights, and every
token comes out as a weighted average.

This notebook is that block diagram, written out. It has five short tasks.
Every one of them is marked with a target, like this: 🎯, and every one is a
line or two of Python. Everything else is done for you.

## What you will do

1. Run a **support desk**: one ticket, five knowledge base articles. Score,
   weigh, blend. That is attention with the names left off.
2. Put the names on. In a sentence every word is a ticket and every word is an
   article, and three matrices decide how each word plays each role.
3. Open a **real trained model** and run your own code on its weights. If your
   numbers match its numbers, you have implemented attention.
4. See what the square root in the formula is for.

In [ ]:
#@title 🗺️ Roadmap (double-click to view the code) { display-mode: "form" }
from IPython.display import HTML, display

_STEPS = [
    ("1", "The desk", "one question, five answers", "#667eea", "#764ba2"),
    ("2", "Every word asks", "three matrices, three roles", "#764ba2", "#e0796d"),
    ("3", "The real thing", "your code inside a trained model", "#e0796d", "#e0a23c"),
    ("4", "The square root", "why the scores get divided", "#e0a23c", "#667eea"),
]


def _roadmap_html(steps):
    cards = []
    for num, title, sub, c1, c2 in steps:
        cards.append(f'''
        <div style="flex: 1 1 150px; min-width: 140px; background: #fff;
                    border: 1px solid #e6e8ee; border-radius: 14px;
                    padding: 14px 12px; text-align: center;">
          <div style="width: 42px; height: 42px; margin: 0 auto 10px;
                      border-radius: 50%; display: flex; align-items: center;
                      justify-content: center; color: #fff; font-weight: 700;
                      font-size: 1.05rem;
                      background: linear-gradient(135deg, {c1}, {c2});">{num}</div>
          <div style="font-weight: 650; color: #23262f; font-size: .95rem;">{title}</div>
          <div style="color: #6b7280; font-size: .8rem; margin-top: 4px;">{sub}</div>
        </div>''')
    return f'''
    <div style="font-family: system-ui, Segoe UI, Roboto, sans-serif;
                border-radius: 18px; border: 1px solid #ecebff; padding: 18px;
                background: linear-gradient(135deg, #f6f8ff, #fbf5ff);">
      <div style="font-size: .78rem; letter-spacing: .14em; text-transform: uppercase;
                  color: #764ba2; margin-bottom: 12px;">
        one formula, four chapters, five tasks
      </div>
      <div style="display: flex; flex-wrap: wrap; gap: 10px;">{''.join(cards)}</div>
    </div>'''


display(HTML(_roadmap_html(_STEPS)))

In [ ]:
#@title ⚙️ Setup: run this cell first, then leave it alone (double-click to view the code) { display-mode: "form" }
import warnings
warnings.filterwarnings("ignore")

# attention.py, plots.py and realmodel.py, gzipped and base64 encoded, so this
# notebook needs no clone and no network. EDIT THE FILES, NEVER THIS BLOB:
# build_notebooks.py rebuilds it. The three sources used to be inlined as
# readable text, which made one 24,000 character cell, and that is what made
# github.com's notebook viewer flicker on 2026-09-16.
_MODULES = (
    "H4sIAAAAAAACA7VcC3PbRpL+K1NyXZl0KESUbcXRLvdKcbQXl9+W1t4cpeUNiaEICwQQABTFqPzfr7/uGWAAgraTXSux"
    "RAIzjZ6efncP7vZ0WZqkjNIkyDZ7x2rvgv87Xxi11OWiUOlcpYlR1TC1MDocqChRyWqZbQL1Ki0XUXKlTFwYFUc3pqAh"
    "uQkukovkfBEVah7FRtHfkmAWNJK+FekqnxnALvNVuVDzNOfbT/+pTqoHmVuTz6LCBIqwuUimqygOTU5PjqOEHhKV9LFM"
    "eV6SlmaapteF0kmoijKPMsJ8VfLN+SqZASBjcJFkOi+jWZTppCzUOo9Kg+vLwsSE+oBQU1rNFjq5MrwOYF5dMDcm36yr"
    "5dHSNaM6T+M4XcsSYzMrVzQvNLPrgZrnemnUxd7JlLDSjAeWrZOaohfJ0gB+VCwv9o4BV9HPqfJ+ANcspyYMiX6EJHYk"
    "T9cqI3qU6bWxc96qEc378K+3+MK4yl2lsYbfVnRBBj63A593Drw2dth7O+x957AbHa+MDDyjgUU6L5f6tvdWPf/Xufpe"
    "Fb/lZS/s93kBQDjL06meRnFUblQYYZOmK6YHVkGrEVAnBOpMvZd1bz1zbaKrRWlCpemOvhIWIuIwLoWwHLHXKsvSvKQt"
    "KK6Z2rNSDWUjz1//GqhntPPEvVOT0yXaqgW4BqxAO6KmhrZycJHQ/sV0k9hpk9AjiGcGzF1LQ5uXWKYnHiuiEBxE0CuG"
    "JTxEjgifaMmosLBgDUmGq2enpz/TQg8PDo/w9Z7a/8/9ANwJVsyC0yBGoM5+fXX+y+n5s6eDxqoJ5Xs8T9E6r03JKwWT"
    "QlZIYEGlKa0ZUGa0cUSW6YbYfpU7QgpTCkNmRCtAM3q2IHqWa0N0PVA9RzWS2jBV64jgRWWfn3WoepCG3MREpSlEl+4E"
    "/3HKEH+8fvPsKfh1fLFHzEi65Opib0ASGhooL5IQ/laSSCbRTMfyVc9m6SopL/YuAePkHRHwxalAYa692HtnSM+EooDC"
    "dDUlopFQ51fElXsDN+iD0yjLjSI9NDOxd/Mky9Qs18WClBtJRVES8b3b70yBfYGuvEqZUzNdFOs0D71BT0VPQSTcepQO"
    "w9wUgsalcNtrT3/YLT5WC7qwXNGOYbbbeAgNbwfvJW8t7crz01+x+CQLdJ7rTc9SYXwYHAzUQXCIX/LpckDX76nPUEdm"
    "8iQ7XX7ZmR0kczPcYDvtkZ3RScf2HBrOEyv8dhLXw29YTR9WMz9H8eSyL/T+sNCWgo6upYljsRekxZJyQB9zYyqt5G8M"
    "ACyjZFWK0VsVKx2TlJT6GvYKloNMqVY5kxiUipKblIxZWN2kWYkxIL9arJYa0v7+5MU/TievTl5aLt6zTxB2t7Bkulxq"
    "ALBywEA6GWFo9+RH/n1ExLI31CO5MbS/qxuHjy1xH+P3E2/GQ3vD3q5vPLGgeMYjnuEIDsVnNVkJ2hN28EwC9TItSqiY"
    "erPILBZshlgfaWWVgqLNzzdEKRL156fnk/PTf57TSi/2Xjo+JJre8DYBdMis/UytScULZ4eqXEdEfPg2URnAGgioBr2G"
    "lg89ibFLOFGFmaUEVNYxoI0nmGmCrU/pGeReyNqKWUrc1uuzQo8NNqnUkDCyfLD5Ea3bPfuwXgdIpD1RsbY2InflmUoz"
    "EoEG1ocNtLeEz6L9jUzZYcCWKdkQTRKSTiIr7xabsCL6nQhjJcezcM6mnbNksRNB8sI2a240fLSCZZJ9Plk9uSjhip7H"
    "PoCaaXgDQmoTApRlESLOLIrFvIpHAmc5p90WfyLM9ZrcPKA5z9OleL7G8ggNYFDMlsRDJTHaNa2kUD+d/v31u1NyiXWU"
    "EAceq5zGp8tvYwV/nZy/fn76yko/cYCIeRxhqdYEJuSTWlHHhNNOSfd1olXcvrZt6Ux/1rCp7p30wvUiry7dTIS2RS8c"
    "kfIFAUfwm/rHztpJpILdFUph/z4acbLdhgzUo3q7o2SgKlaB8xY4Rw0Ac5J6XqGlO6GhV3E5oes9PL0vwz68taPIIwh7"
    "dDNI0nyp4x4YcdRjSgXFQmdmPCQjQS4wLdXNff5vzH3/5+fmhgiQEOoDQoH+vf9GwgqJmJM6VFlkIA3g/O2AK1CnsIVw"
    "GNkpR0CnEAKu02/B7N9gnb+YOAML+cEnFDTkH9vDN0pdXBccINfBxaZBjm+xWggPGaBwQq7ada9oSgscvwXwEck6VuRK"
    "MK5WjZHg4NuUTKLYlYaArIUBfzd5WkygKXokXmG5ycxoHqe6tMy2HrOWuEI0WPQvaRJJeoMP17WcLygkJBE/a+IpXp/m"
    "ZMP9wss+XKUQaRv9kTmEBRbdDX/WJiuWOgtcHP2Onyg7xaGbXam2YWVGERyWTUiYeD5wFIDrp/S8RMKhxFWBVt2aGrLr"
    "hu9BqdtZ8ygvJHaKNX2YpfFqiWenV4Zu5wPS+tGM0x0CDshWFg2Wg9lmndIK8msX24NLasrI5wRRshX2g8sGbe/km8zA"
    "mi72jhVvT4+2BSTojc/G0UBFl+KbIJeTw4ntJf1LUhr+/MTcljvnq+/UcBuG2iet04aT5eZmN5z9DjjDgdpGR8hS1JDO"
    "xsdkPS4FXJ8Q4gv7Q3fFTf9UMxzxyYQIl2cJuehtvoP+ijkucTICjNhzp3mBGjIPFbxx18Y64cKAHNo27YlsyFkAQdC3"
    "UTHaH3oWjv2KXqKXZkBsTYy01ggEdJnGo6HZf9jE7E0eJaV6c3J2Bk356vW5+pVcSXZIODCXDEhQ8ft5vjJgS8yohAHL"
    "M7cZmUiSHZnBbMc6Cx5rKv5RaGYRWRaStAWzJrNlGpPQ0bcOdiT0rW9QiHfA6+nQDLTC5kBZ8/ZIUqcjgBUeV6MRz7Xf"
    "IGCAEcezOC1Mr009/LJwItII18c1D2UgY4/ICUKqu08Xe8EcVrTkjejbWchldkxyVN85zx9MHzdEy4L+HsOhqiw3Y/uo"
    "HxCSUVH2dsx129ScK4vcnmxZLb2+SGj4XhanZFz8hO4p68h5dIXUpE2aNbKtzooF8JwwP46mXjqXDAXxPDPSLykZOjJj"
    "saHJA+S0IqszjSF1FZKQVPlY+O2kBcFTteCwpBB5s1We1alhSQ2tyd9Ol6SUIRTEIO6ujkkVJ7qEb2HHaNa/LAIk5Ro+"
    "HetNC461O9g6t2nZhKVDnPCrHCkpQCBUObL6YtLOXqupQ/TFJ9zP4vIiYVffu4110/bbeS/Ix9H5mblaQt+HT3GT7BSe"
    "9uYf7968OB0o+XvI0dm9o6MfjNHiid/74ejRVB8Ct5OXP52+kxHmQB8+nOHi/7w7+VWu/aj1wykPfPbquVw6fHh4dDiX"
    "dSETwQnvlETcZEp24FgoXLGCKD7Sqxg3JML8cnqCmHHHEgIsfMLseLG3AHNMBC6Qp9ji3px/8K21UvH3iXhBPnujc70s"
    "aLjwaBBmEYUe8B2GB1tj9C2puiJDHYAkIZOBfyeFZT47NMe6vjj4Ko/Czw2ap1BE5HPLoB9rjU52hba6F2uKH0l3JqPD"
    "J06JWwEdF9BIxIvkGKm/kiaUskkxPk7ECn5HW0ZGhK0hC5YAu/yGwXVHnvgbuaVEHrK9kq4AfUhz0p8yKmMzutj7hRy4"
    "tSF/NExt5GAzOCRTZH/99Nl/X+w1reMvaR79TvuiYzXVOVetKAa3M+WJA/FqdWnVQ9aw1eTTkaKw8e1VgW0srG4lhiQt"
    "d0s3wQjFasratUeXJfD6AUmQI6eJ9W1AGCx6DIg9GdltCsaKMT+FYjNWDSORgnpeYcrJBmsutmd3jBLO6FmmGzM1x75T"
    "J09zU3F5M1A37NCQZiNjXpIDb3Hqe+aOHlKS39e7IW6kyJxi7w200N1xcDj3DN8NLelG08bNoA5yyLesi1RPE93bOFr2"
    "KMyXgEA9IJY7fNwegtWQAuHNUr2QFKvNxvRpsxtjmWF6/Hug4nRGKMRmXlbDsE0lRH0SazLBZCXr62DBnueEMUu6PANK"
    "WAMOnixzNpnsjG67yAnMJIUfwxMUoi2pgdD2k3WMcj+J2+C1jeOzeoMJcv+PsNuTNrtteK8OHfYLxnJ0EDx84nYFRmIg"
    "6oQIVuGseiwWyCs2CG2h7gtUS5ltqE6jO7iFT6OerQlu7aDl8s0XuVp2oR4WJeTH0Eh40z2fsR1n/x5lvY3g+xmWpuBi"
    "U62NOfvgv/4tzh4Gw8MdDO2sKTgFMaoOw0KtMratPl1ic0XOU8/yc7pGUVTsla0ip8mIDVKXMFDkQlHjWm8QbKoiMxyT"
    "GvJ/yENCBbQqNu/9Z0RmGgNZkHlCbt0asS0nHdwXDi4mHUIEU8OT2RPDYIUgU9lKPoDsM186SFxW1ImJWZ4kd9cQJysw"
    "ZDNaIkO7DOHyURmoWo6Cx5CkR75+1Lci+ASPzIXjJ8BurEj4q2vhLZ4jIeqNwR6k88dTglrp/rGIo4gPXV1HYbkgyTrq"
    "NwAwJ4mwCJwdt63UjD3BFteRSU2OyvY8YRxZbQdHOJLcbpkMWYi/Ul/CML7bWsDzuVF/VYfi8/CQYcugNGG6n0VTFlk0"
    "Kegt02WXaPoaxTM82FGxPQ+RuDD7P/5Z3q+zB7UTM9ezkrz9UY9UCvEVsd4j+k1ewRbzF2gIcS7JchWXURZHUk/XFJek"
    "a4RBAs5lmbh+IKq0yfmIOGq/3UYaduTXCYfF2xeM4YE4NMTltFCzGSGV0N9putryM2+KjXuAxy5rr1ukoB2Z99tCs4AO"
    "X3f6SV3KdxdzW2+iULfq7vjK47R5fyfPd6tvT1GTk3Vw2W3FWrc+a8pkbNOecUEZZSOJyIghblHOLWQjQi5U4YY4q+zc"
    "mqLhwlSbThudOSL8FF1d0XjHc8y++G45Sr0UJtyQjKerqwUHyBGKYBRlW0+80ipB1WzQ+LkdiVVlSRWaotSWlMxSPwaP"
    "/6Cs/ecDnUpKKGItvmV8Q/B7Z4Oq3uiimwYCoI6+Hb0i24b8cYI+MsNyRjxP/C8i14xwjKbdytrNY5zrVu/QfIZshq33"
    "DqrUMy5em03RDHbWyB+Tj0msBRzqVNk68eQUe0fjDoJHj0lKIeyyKORZh8GRZye+4LIyoO8U6cYneApWKDbgAEqGbjqx"
    "iJaMVxAtmRmIjDNa3wj5B1L7yygZHeAvkW7Y8ra6wiaLbf8rIqzuob5pdfuZ256/0dGBZfe8pSC2FIAF3u0eYnOOq7oC"
    "4gro/zilOeQgbcN18+xO11MpbObeKuS6aDb3Nf2xuCmaV7woNQxvx/86Imt23HQNvGx9XYto+wUY+HF74HBrIH5uUNZA"
    "TeDj5fZNdh/+NuImkONuR8E5IR8HKtoVsnY5E9XXbrD1WqxC+8GZJrIPSKhd7FnsCLnHj4W3a4fEsTwXihoSBnFhQHAU"
    "oyUrBbahtlUUWvXR0YD835A/1nFWyyxtye4uRdutbOv+JHObxTpKiPdQeoOCYLaSFJErnMMutJyPQOp43PYnLbQEZimp"
    "4xlzlQrziGJDTUFK3RUBjWYjaXSbINP48+TV63PuKrpz+s9VoYjtP1ijx4VXxA2NBjhVl4ok4Z0bHYt0oFUE/VOFWi/S"
    "2E932sgDcYZTkB6caLk0YUQiga6hqh7I7V46v5ZeY5lclCZzwHKpps67IPLcSF+liY6P1R1WwfEnqjW2sCnkRiHQ3M5M"
    "Jt3Lrj7YAdKWC3M2A1WCixmOy7nsK7vq44IeopK0Jl8nkilrEYJWd3hLI2Ctquq2afgiJTiDW6O74HEnOLMUOl1io3Pw"
    "WJYWERYnRdTE6yQne9WvuhpdFZP2/4V0eZe2Xtu183e4YSk6d32LpMUadeSY2UHqvmwPxWAOvrRhKGjwMlD+lrbfpbSU"
    "oWGQm6+kdBynoBh37HWAXOti4PwsEAZ5AcJpQ5t4bUyGzK+oLOBKUdeCqxhk+JfRLUjUxahlg2bIQFmanUeMV5an5N8x"
    "v2PR8Ee7yHf/Tn+6j6o00edOTytCYk5DaO7fTT/dl1XwRzunA7W7qRYwU00eZMVPtmzkmMCVpcD2wmAF8hndG7LhHi3u"
    "JXa1dXaOu7isIHXlc3CXABmdsJp7VlYVp3zW7OfPzbpqZmb8GvT2StMoL3IdqiaX7Q+oBnUYGiKTvV2RvFoOsa6tRld8"
    "6yi1tRhfIVReoD0JQUYIFp/3DF9ySU2Nn744u+SL3dDGZ6dvLp364NZAjOXaq22+cE0MAfp1k6q5AruQ1LstrS+rWKOz"
    "QfRL9wM1q1iAdyvgJjk+AnA/SdfSE+wDvK+mKxQLgh0QT7mcyskxIwc+MB+iKzYCUYHdz0+tcB8bMLnScUwL7eWrZCCC"
    "Clcw5mYUt8uTeTLgzO+o53XU6ThCD21/uxHHdWDjAcct+SJ8RLQq62jjsCk0RsmYFVVdnzGRXqrCkGeYyOGanpzVIRez"
    "z3gV2EMjEQEtPzTIC9Qmt6Ow351hEFtvPQ9NvtaUrDVRJrD9Kgy6hyfCIRzsuFUls3bmKOCA8tIIxqGfoRgid/cwcGGJ"
    "jOl/wXtTKOwVxAaT6/XojnQwUm6THDqWxXY85Dzu40vs2qLINLbtmD26T17YbjMg5ETropwchpzn8HIgtKuDBuUbyTNB"
    "1XPUKMaZIOjxmAhMFiyiMDTJmFmNMHrA18CJrlhhmRDX/BYCPwat6c6tnWxe7j6JgvRbGHxQXURkr9QUY14aGt+rkJUT"
    "0jZm9ZPTVY7DTgkQDk0yLqHa+ij70buG3yLzEfBADl0wuvdk1+jNrtH1eBv1TNMQKaya58e0RV6s4TzcUd2XtnUzWGUh"
    "tlKPLIcT2JHlaCLNdEQRDAsFbmh8m9I33UzBcpzrljC8bOSfOBl0sZeiZt7frmG4zlm7pJICqwIbOcIAfDmRxJsLVoaH"
    "JC3bu4r7wkucTY3DOpdqy/M2OOICezceFNA/YZcJzRPBR4oUepN1TswHOjvuevDAEo6k+NFRp5h+eQ0HkPgq0+tj1gGO"
    "DwiS/JLZGZGqePwnEr2yDKzT5uab6vsnMhrXiHJ0JooRJxHlXCISNOxm/B/P+z8+DkDhHDin0UwJF2Igk+QP8QQeGBRZ"
    "HBFyZJdZFVW6mJQL+5aR9T88LWKbCgAF2Rl8xhDO1FBUypi0YmZ+cKCzjKtOmLh9nzCqz8B0NEa1xgENRHp8yZ4HTQz3"
    "NOD/7zxgQJjuebB24WM7J/h2yzLTJvCByIZVFs3Laq9pm7dT8dzr5oC45lE3s/KyrANIPhg3dXItrja95+zo9GC9qxOa"
    "/cHWucvGFe8kpABpHIfknlY8GKmhwZdOP9q+v6813bZD3m5Cl9nmL1Wru+sXJ1VGn6f0efreztppjdwBVjvMN2T2nOpA"
    "ESDAsej0Tgc7nmWBkTE7aVvJz835cvHjke9PPIQ3cfBvOxCHckiM/Ygn/IkFeO35Eg8bvgRKAEj91MnIuoOfmOh0HF02"
    "y/dSJiT131FzcOn+YWB55v7dp/ukmcniWwVcW3zL5I2ndWQEffCuDHnp3xu28X/rQ3zr4e8aBTpXMGyu4DDwTyx3r+Dt"
    "16A+7EadMAJLjKWZjlTRRzSYRqKyBNF23pLdzEpU+v1L7ymHbSK0Bw+Q1Kwogaf3m9N3prB9KLvm2Cyz7+ztTFJ7JvWH"
    "DnibVkXNv+c25yFvjmvxaCgr2irX5WBjed2ueuzaqYdtGp74+3uyxUiHOzjpYRPZR0G35iRMGyfHB6qLx06+hscedvOY"
    "lEPBQBjVakXxnGHuC7/Y42PH247xH69No8g3QRm5XPTCiVvdkoIVsmHhpCjDuib1Tq9VmJb77tibrZZitnSWh+0+u5cW"
    "jiI42OO3wXOlr3SUFBKf2kP/NkJdxaF9NQLId7VP/xouUEaQIjQ5Ay2JqwSAxfsrO6KO0BFdlzpwd+fSya3ebzu6dVZ9"
    "2V6eCtRzv4zSgt1YAIDve8CbDVeWMu2azG0x08ypRBtMnerCjFqtRJuvGFPVk9C65h0iZhX6Pd7osKuQVG4ynHOXap8V"
    "CvROMjP80RJS3cfU1bL0J7hZ4yzuxMzn8BLEb7D6sMGX74hjq7YMe0wTbzegD3DFw+gmgmOGJgu7EXzSh1M/b3no86/p"
    "rHAVdL/DAozrjNG+i+RAwJE7mQu03QEIi1DdAEEjvXuT5h31fS0T/f6XnRo/SfKj5EiqTsEqMWBxqEvU8GQELUtvW562"
    "1r8d7nfBm/jQmNrS0cKo333qo5rT6MAIB/7K6scNdz3Ot+cV7/6h/jX/gEXrjI99A8lAtQjROrlh79bneYb9gEKCTWZ6"
    "cnaFdmDraMbXPLJ63tZhESbu1z8Qhz/A1Ms0NHHjAMgJMzurLD5gTPvDgwac0KVvq8y1l/gnFO+7QxD122rssWjyN1z9"
    "xDkDnBWW09KVVXD1lYYnUATuVS2MAzKXP9MSopj8sXPVC/kz6bByH7puf0XPLEzYP1ZHEuURmw0POdhgLXeR2PDjh6Mn"
    "De1n3dyjR1We1WX4iyVOPtqGF05G65BP+T1Nibko1nv65h8w3nKARNsXMHglG/Z5ooTNhCxCDuNjvbRwEoCLBNlYneMd"
    "KOktTm1p19E+o0iyiOYRv87I9TmjigqynFanyaoXEGUxLYqPhPOKSL45uScZ4mwVx/X5c3CvLuzLmfCKInmPk1QU5ZVO"
    "/IoI5b/46SveWPPy9c+nL/iFEXysZMcWCYQJD6Zx0lwiJ82f/S8fXJFLTsuD7D0mH3dWjuqnNPX7C2xPdayH1p9XxU0h"
    "Prqm+K1UBrZZ6ma4g5fOqDBPM6JLv6Hir2IKwmMluA5UjWSVoLDLsE0yfrZFKFRlreA01p0RXNv2brnhJ6syfSkih4/n"
    "biX11HvqLIqrA3fMkrnhuZJRWiX8DoibdKankwd1m3WzHubDk6JTAYGMya1eIWkgBOMkMLNshZccvEACh9+11Xck9gEu"
    "VhAOW90hxRaon3D6UN6ilC9jmF+2vOz8IbcCFjJ5zkehEuH8wMvLeoQKyL+5QhUOOp6EYIrC4GbCk/3+hTLftHJQri6x"
    "4vlzirYnhGhzTOtmsCL+/epHShlenbpqfOv5qP3WVxr83thqOSpVq2CP9b2HVeJT7cxn5nGOK5kQBWKDc1kS913sGdrq"
    "vJG3FbABBMRf2j3ibsXBiCm5GVve83KV6zDiN9NUb8cgFsxWpZSZuEtEkQYpUhZFH17AqqPX56JdwW+6MC5MLIhBCMH7"
    "BT9ARDTyGcJimZvfVhE5TxMMm/R8T8AmBGsqD+wkUSvMYerdKtmqt1UnrctFLnrfqQ9rhetzvDhjCEpQWJs19Kl9SVkp"
    "6bpGYCcO0HH9xjaREmT2+KVtUmWtj4hzVZVzijWM0+Odb31TcxPaPDMXcg64kzODhmPD522ppNyOXV61wNENycK9uHS2"
    "K0qwlxW0F7uhgb8qtPjUrkLINSRnM8Eh7UFNx/tdtr6ZmrROM1LskyiJysmkJwfv3e4M1GftgaW0HUVC0jIhvtAmM6TU"
    "0+teDVuYZyKMi8A7a7b/inPO4HoPHtCcgWX7SbWowjZq2utCWQoAicsLv1eanWRaWlDx3ahaZGuEsI4gG8xSaQqOwmJS"
    "phO516NZ44s93jXcudi7RG21BUdwQWprgbSdk0PkIhZ8Ioyc1wa+ly0Ap4xjBal6t0A1AMxgI/VSz657Y91+kHYPqgm2"
    "hWcycW6c/Squ3Kh+hg2ojg/bCIQ81u0RiIVu5zBaqu+/b16zUFvzJ45v+K/Pj418tvCkn9U+9l9E4CoI8tau+uU/bDBx"
    "aRrpwlQ1BalJbKkR/LzZnKdwEYuScy/yMhT0abBY0hTUK2/Vh3+dq+/UVPTUB8BlAtVwehzSRUmfPD3Y1Cwt0DGOeejC"
    "sMJNki3i/RcWWtddUb2QgrUH98+ISX9ArjPpiN7iu2GfPgfqLJWOKl7yrRUnw42CAvzoUT9oU6qhS9wuy0YEngMQ8Jpt"
    "pSDwIo6qRJRCkaE4CxQf+AxhS+6od/Wbd+rp/OIeAhv8NiEiB7LVjneDc7w5Ik6PF5HHcvy+Hky5/vop7+2Um6+eMm0i"
    "Bt5xE8ZbY5sYfX5sE5XPjt16Q5BfUfHFRFQtAv+vlJGGNfBcdungc8f9Q1YeTliCNuNY9Cr9MPYee9mwKlLFEtyklrWF"
    "1Ruv40y7ING+22tJDgq8ZO2HwfwqMLxFFCFaCzMKFKICoS7ejeEKK2Sn2/3NdgXe61rFVJdcirjm5ZNZK1vnc2vzwAeo"
    "SlQq+PtlAwV47YDVfqiOCqPeI115yj5tV28U6kQQYMDABrg+PDFKd58CKXJKByqfTESsJ8GnpkfkZkfTFdeElxqxieKi"
    "tvgv9+/du29fEBWopylFmXiquSUCx1VvsC7Zcwm6ILeqWD6N+ls1cVClr/6mhm2f3eZl8K6KY66VqXQ2wws77j7xa/CK"
    "v6hV4VrqxW2T8xpl3bII4nRhiNW/gR+q67GSmEByoOTYK1X2nGOK/sVgqzhXIT/gnYW5728JhL3jSwC3eE0Qy03IvUKX"
    "3bak7hSNv1shtDi4yg6SHvymJO6Yq14WZc/UM3G2BaPS9NuFZWlaW/sWf9xAz1dNHWfzaW5/fHy8P9zWYGNfZsYfLwf2"
    "VUE0hb71+3VlT06++6TDasRrmNAoX4nQH+Tvym2KfeCuyEYBXJrtuB/W6ruaaMLeHBmgV5yBbpNugJJkB/UGzWuCUutk"
    "XUVQsjTHfBDiYzcxQ3Hh6NMqyfUNaXUB69F5PeAzaHxSyRKcNKU4Z/1+wz2syA+xetFnHdhb9B3512MKMRaO/vjsb8He"
    "p/8H5jjooIpcAAA="
)

# The modules are written into their own folder, never into the working
# directory, so running this notebook inside the repository cannot overwrite
# the source files.
import base64, gzip, json, os, sys, importlib
_SRC_DIR = "cx_attention_src"
os.makedirs(_SRC_DIR, exist_ok=True)
for _name, _src in json.loads(gzip.decompress(base64.b64decode(_MODULES))).items():
    with open(os.path.join(_SRC_DIR, _name), "w", encoding="utf-8") as fh:
        fh.write(_src)
if _SRC_DIR not in sys.path:
    sys.path.insert(0, _SRC_DIR)

# Drop any copy of these three modules this kernel already imported. Without
# this, rerunning the notebook in a session that ran an older version keeps
# the old module object, and a newly added function looks like it is missing.
for _mod in ("attention", "plots", "realmodel"):
    sys.modules.pop(_mod, None)
importlib.invalidate_caches()
import numpy as np
np.set_printoptions(precision=3, suppress=True)


class _Blank:
    # The blank in a task cell. Any use of it stops with a clear message.
    def _stop(self, *args, **kwargs):
        raise NotImplementedError(
            "🎯 A task cell still has a blank (___), or you filled it in and did not run "
            "that cell again. Run the task cell (Shift+Enter), then run this cell again.")
    __getattr__ = __call__ = __iter__ = _stop
    __matmul__ = __rmatmul__ = __truediv__ = __rtruediv__ = _stop
    __add__ = __radd__ = __sub__ = __rsub__ = __mul__ = __rmul__ = _stop


___ = _Blank()
try:  # stop IPython from reusing the name ___ for its own output history
    get_ipython().displayhook.cache_size = 0
except NameError:
    pass

import attention as A
import plots as P
import realmodel as R

NOTEBOOK_VERSION = "v1.0"

try:
    import transformers  # preinstalled on Colab
except ImportError:  # anywhere else
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "transformers", "torch"], check=True)

print("Setup complete. Notebook", NOTEBOOK_VERSION)
print("Support desk: {} articles, {} topics, one ticket.".format(len(A.ARTICLES), len(A.TOPICS)))

---

# Act 1. The support desk

A ticket arrives at a support desk:

> *My parcel never arrived and I was charged twice for it.*

Behind the desk sits a knowledge base of five articles. The agent's job is to
find the article that answers the ticket and read off what it says: how long
this kind of case takes, whether a refund is likely, whether a human has to
step in.

To make that job something a computer can do, every article and every ticket
is described by **four numbers**, one per topic, each between 0 and 2: how
much is this text about billing, delivery, technical trouble, the account.
These numbers are made up for this exercise. In a real system a model
produces them, and in act 3 you will meet such a model.

In [ ]:
#@title 🎫 The ticket and the five articles, as numbers (double-click to view the code) { display-mode: "form" }
print("The ticket:", A.TICKET_TEXT)
print()
print("{:32s}".format("") + "  ".join("{:>9s}".format(t) for t in A.TOPICS))
print("{:32s}".format("ticket") + "  ".join("{:9.1f}".format(v) for v in A.TICKET))
print()
for name, key in zip(A.ARTICLES, A.KEYS):
    print("{:32s}".format(name) + "  ".join("{:9.1f}".format(v) for v in key))

In the words of the lecture: the ticket is the **query**, the five articles
are the **keys**, and what each article says is its **value**. The desk has
to answer one question: *how well does the query match each key?*

## 🎯 Task 1: score every key against the query

The score is a **dot product**: multiply the two lists of four numbers entry
by entry, and add the four products up. Two texts about the same topics get
a big score; two texts about different topics get a score near zero.

In Python, `Q @ K.T` computes that dot product between `Q` and every row of
`K` at once. The `.T` is the **transpose**: it lays the table of keys on its
side, so that the four topic numbers of the query meet the four topic numbers
of each key. Without it the shapes do not line up and Python complains.

Write the function so it works for one query or for a whole table of queries,
one per row. The same line does both.

In [ ]:
# ============================================================
# 🎯 TASK 1: the scores
# ============================================================

def scores(Q, K):
    # Q: one query (4 numbers) or a table of queries, one per row.
    # K: the table of keys, one per row.
    # 🎯 COMPLETE THIS LINE:
    return ___ @ ___.T

# ------------------------------------------------------------
# Hand your function to the rest of the notebook.
A.scores = scores

In [ ]:
#@title ✅ Check task 1: two PASS lines (double-click to view the code) { display-mode: "form" }
s = scores(A.TICKET, A.KEYS)
A.check("scores of the ticket against the five articles", s, [3.24, 4.28, 0.08, 0.4, 2.48])
A.check("two tickets at once, one row of scores each",
        scores(np.stack([A.TICKET, A.TICKET2]), A.KEYS), [[3.24, 4.28, 0.08, 0.4, 2.48], [0.04, 0.0, 4.08, 1.2, 0.2]])
P.show_scores(s, A.ARTICLES)

Notice how in order to calculate the scores in the cell above we multiplied
Q with the transpose of K in order to line everything up. As mentioned, if we
do not transpose, Python will complain as shown below:

In [ ]:
def scores_no_T(Q, K):
    return Q @ K  # no transpose

print("Scores without .T")
try:
    scores_no_T(A.TICKET, A.KEYS)
except ValueError as e:
    print("   crashes:", e)

"Where is my parcel" wins, and rightly so. But look at the runner up: "Refunds and double charges" scores
well too, because the ticket also says *charged twice*. A desk that picks the
single best article and stops throws that away.

## 🎯 Task 2: turn scores into weights

Instead of picking one article, spend **one unit of attention** across all
five, giving more to the articles that scored higher. A list of weights that
are all positive and add up to 1 is exactly a probability distribution, and
the standard way to make one out of scores is the **softmax**:

```
    weight of article j  =  exp(score_j)  /  (exp(score_1) + ... + exp(score_5))
```

`exp` makes every number positive and stretches the gaps between them.
Dividing by the total makes the weights add up to 1.

Two lines are done for you. The first subtracts the largest score from every
score, which changes nothing in the result (every weight is a ratio) and
keeps `exp` from producing huge numbers. The second takes `exp`. You write
the division. `w.sum(axis=-1, keepdims=True)` adds up along each row and
keeps the result in a shape that divides row by row, so the same function
later works on a whole table of scores at once.

In [ ]:
# ============================================================
# 🎯 TASK 2: the softmax
# ============================================================

def softmax(z):
    z = z - z.max(axis=-1, keepdims=True)    # done for you: shift, result unchanged
    w = np.exp(z)                             # done for you: every number positive
    # 🎯 COMPLETE THIS LINE:
    return ___ / ___.sum(axis=-1, keepdims=True)

# ------------------------------------------------------------
A.softmax = softmax

In [ ]:
#@title ✅ Check task 2: one PASS line, and the weights add up to 1 (double-click to view the code) { display-mode: "form" }
soft = softmax(s)
hard = A.hard_pick(s)
A.check("softmax of the scores", soft, [0.2274, 0.6433, 0.0096, 0.0133, 0.1063])
print("the weights add up to", round(float(soft.sum()), 6))
P.show_weights(soft, hard, A.ARTICLES)

The gray bars are the hard pick: everything on one article. The purple bars
are the soft weights: the best article gets the most, about 64%, the
refund article keeps a real share of about 23%, and the articles
that have nothing to do with the ticket get almost nothing.

## 🎯 Task 3: blend the answers

Each article carries three numbers, its **value**: minutes the case usually
takes, chance a refund is involved, chance it needs a human. The desk's
answer for this ticket is the weighted average of the five values, with the
weights you just computed.

With the weights in a row `S` and the values in a table `V` (one article per
row), that weighted average is one matrix product: `S @ V`. Row by row it
reads: take 23% of the first article's numbers, 64% of the second's, and so
on, and add them up.

In [ ]:
# ============================================================
# 🎯 TASK 3: the blend
# ============================================================

def blend(S, V):
    # S: weights, one row per query, adding up to 1 along the row.
    # V: values, one row per key.
    # 🎯 COMPLETE THIS LINE:
    return ___ @ ___

# ------------------------------------------------------------
A.blend = blend

In [ ]:
#@title ✅ Check task 3: two PASS lines (double-click to view the code) { display-mode: "form" }
answer = blend(soft, A.VALUES)
A.check("blended answer for the ticket", answer, [6.4338, 0.2854, 0.2517])
A.check("soft and hard weights in one call, one row of answers each",
        blend(np.stack([soft, hard]), A.VALUES), [[6.4338, 0.2854, 0.2517], [4.0, 0.1, 0.1]])
for name, v in zip(A.VALUE_NAMES, answer):
    print("   {:15s} {:.2f}".format(name, v))
P.show_blend(answer, blend(hard, A.VALUES), A.VALUE_NAMES)

The hard pick reads the parcel article and says: 4 minutes, a 10%
chance of a refund. The blend says 6.4 minutes and a 29% chance of a
refund, because it also listened to the article about double charges. The
blend is the more honest answer to a ticket that raised two problems.

**You have just run one head of attention.** Score the query against every
key, softmax the scores into weights, blend the values by the weights. Every
attention layer in every language model does exactly this, thousands of
times per sentence.

One more thing before the names go on. Watch what happens to the weights when
the scores are multiplied by a growing number before the softmax:

In [ ]:
P.show_sharpness(s, A.ARTICLES)

Bigger scores give a sharper softmax. Multiply by enough and the soft weights
become the hard pick. Keep that picture in mind; it comes back in act 4.

---

# Act 2. Every word asks

Now the sentence. In the lecture's block diagram the input is `E`, one row of
numbers per token: the **embeddings**. Attention lets every word look at
every other word, so every word plays all three roles at once. It is a query
when it asks, a key when it is asked, and a value when it is read.

One row of numbers cannot play three roles by itself. So the model keeps
**three matrices**, and multiplies the embeddings by each of them:

```
    Q = E @ WQ        every token as a query
    K = E @ WK        every token as a key
    V = E @ WV        every token as a value
```

Those three matrices are the only thing here that training changes. Before
training they are random numbers, and that is what you get below: a toy
sentence of three tokens, four numbers each, and three random matrices with
one decimal, so that every product can be checked with a pencil.

In [ ]:
#@title 📊 E, and the three toy matrices WQ, WK and WV (double-click to view the code) { display-mode: "form" }
WQ, WK, WV = A.toy_weights()

print("E, the embeddings. One row per token, four numbers each.")
for tok, row in zip(A.TOY_TOKENS, A.TOY_E):
    print("   {:8s}".format(tok), row)
print()
print("WQ, 4 rows in, 2 columns out:")
print(WQ)
print("WK:")
print(WK)
print("WV:")
print(WV)

## 🎯 Task 4: read the embeddings three ways

Three lines, one per matrix. Real models add a constant vector (a **bias**)
after each product, so the function carries `bQ`, `bK`, `bV` along. In this
toy they are zero and change nothing; in act 3 they will be the model's own.

In [ ]:
# ============================================================
# 🎯 TASK 4: queries, keys and values from the embeddings
# ============================================================

def project(E, WQ, WK, WV, bQ=0.0, bK=0.0, bV=0.0):
    # 🎯 COMPLETE THESE THREE LINES:
    Q = ___ @ ___ + bQ
    K = ___ @ ___ + bK
    V = ___ @ ___ + bV
    return Q, K, V

# ------------------------------------------------------------
A.project = project

In [ ]:
#@title ✅ Check task 4: three PASS lines (double-click to view the code) { display-mode: "form" }
Q, K, V = project(A.TOY_E, WQ, WK, WV)
A.check("Q", Q, [[-0.2, -0.1], [-2.2, 1.7], [-2.7, 1.6]])
A.check("K", K, [[-0.4, -0.3], [0.9, -0.1], [0.4, 0.3]])
A.check("V", V, [[-0.9, 1.2], [-0.3, -1.5], [-0.5, -0.9]])
print()
print("Q, every token as a query (one row each):")
print(Q)

Pencil check for the first row of `Q`: the token `she` is `[1, 0, 1, 0]`, so
its query is row 1 of `WQ` plus row 3 of `WQ`. Add them and compare.

## 🎯 Task 5: the whole head

Now assemble the block diagram. You have every piece:

```
    Q, K, V  =  project(E, WQ, WK, WV, ...)
    S        =  softmax( scores(Q, K) / sqrt(d) )      d = numbers per query
    A        =  blend(S, V)
```

`S` is the **attention map**: one row per token, and each row is a
probability distribution over all the tokens, the weights that token puts on
each of them, itself included. `A` is every token rewritten as a weighted
average of the values. In the code the output is called `A_out`, because `A`
is already the name of the helper module you have been calling with
`A.check`. The division by `sqrt(d)` is the scaling note on the lecture
slide; act 4 shows what it does. `d` is read off for you.

In [ ]:
# ============================================================
# 🎯 TASK 5: one head of attention, start to finish
# ============================================================

def attention(E, WQ, WK, WV, bQ=0.0, bK=0.0, bV=0.0):
    Q, K, V = project(E, WQ, WK, WV, bQ, bK, bV)   # done for you
    d = Q.shape[-1]                                # done for you
    # 🎯 COMPLETE THESE TWO LINES:
    S = softmax(___ / np.sqrt(d))
    A_out = ___
    return S, A_out

# ------------------------------------------------------------
A.attention = attention

In [ ]:
#@title ✅ Check task 5: two PASS lines, then the map (double-click to view the code) { display-mode: "form" }
S, A_out = attention(A.TOY_E, WQ, WK, WV)
A.check("S, the attention map", S, [[0.3736, 0.3065, 0.3198], [0.5679, 0.0956, 0.3365], [0.6524, 0.0684, 0.2792]])
A.check("A, the output", A_out, [[-0.5882, -0.2993], [-0.708, 0.2352], [-0.7473, 0.429]])
print()
print("Every row of S adds up to 1:", np.round(S.sum(axis=1), 6))
P.show_map(S, A.TOY_TOKENS, "Toy attention map: random weights, so it means nothing yet")

Read the map row by row. Row `she` says how much `she` looks at each of the
three tokens, itself included. Row `she` spreads its weight almost evenly;
rows `liked` and `tennis` lean toward `she`. None of that means anything: the
three matrices are random, so the pattern is arbitrary and **nothing has been
learned yet**. Training adjusts `WQ`, `WK` and `WV` until the maps become
useful. Which is the cue for a model that has been trained.

---

# Act 3. The real thing

DistilBERT is a trained language model of the kind that sits inside search
boxes, ticket classifiers and document filters in production. It reads a
sentence as a list of tokens, turns each into 768 numbers, and runs 6 layers
of attention with 12 heads each. Every one of those 72 heads is the function
you just wrote, with its own `WQ`, `WK`, `WV`.

The cell below downloads the model (about 260 MB, a minute on Colab) and runs
the sentence from the lecture slide through it. Then we take the table of
768 number rows that arrives at one layer, and one head's three matrices, out
of the model, run **your** `attention` on them, and compare with the map the
model computed itself.

In [ ]:
#@title 📥 Load DistilBERT and run the sentence through it (double-click to view the code) { display-mode: "form" }
SENTENCE = "Alice tried golf and tennis and she liked them"

run = R.Run(SENTENCE)
print("tokens :", run.tokens)
print("E      :", run.E.shape, "  one row per token, 768 numbers each, the input to layer 0")
print("heads  :", run.n_layers, "layers x", run.n_heads, "heads, each query has", run.d_head, "numbers")

In [ ]:
LAYER, HEAD = 1, 8

WQ_r, WK_r, WV_r, bQ_r, bK_r, bV_r = run.head_weights(LAYER, HEAD)
print("WQ of this head:", WQ_r.shape, "  768 numbers in, 64 out")

# run.hidden[LAYER] is what arrives at layer LAYER: one row of 768 numbers per
# token. Layer 0 reads E itself; every later layer reads what the layer before
# it handed on. This afternoon's lecture stacks the layers.
S_yours, A_yours = attention(run.hidden[LAYER], WQ_r, WK_r, WV_r, bQ_r, bK_r, bV_r)
S_model = run.model_map(LAYER, HEAD)

gap = float(np.abs(S_yours - S_model).max())
print("largest difference between your map and the model's: {:.1e}".format(gap))
if gap < 1e-4:
    print("PASS  your attention and the model's attention agree. You have implemented it.")
else:
    print("NOT YET  the maps differ. Check tasks 1 to 5 and rerun this cell.")

Now look at what a trained head does with the sentence.

In [ ]:
P.show_map(S_yours, run.tokens, "Layer {} head {}: where each word looks".format(LAYER, HEAD))

Find the row `she`. Four fifths of its weight, 0.80, sits on the column
`alice`, and the row `alice` returns the favour: in this sentence the head
ties the name and its pronoun to each other. The head learned that from text
alone. Links like this one are what the lecture drew as arcs between words,
and the five lines you wrote computed it.

The lecture's last frames read one token four ways as it passes through the
head: as a token, as a query, as a probability distribution over the keys,
and as a weighted average of the values. Here are those four readings for
`she`:

In [ ]:
P.show_readings(run, LAYER, HEAD, "she", attention)

This layer has twelve heads, each with its own three matrices. Same sentence,
same formula, same five lines of yours: only the matrices differ, and
training chose them. Four of the twelve are worth a look one at a time,
because each has picked up a different habit.

Read every map the same way. One row per word, and that row shows where the
word sends its attention. A dark cell means a lot of weight, a pale one
almost none, and every row adds up to 1.

In [ ]:
# (head, what it does). Every percentage in the captions is measured from the
# maps your own function produced, so none of it is typed in by hand.
HEAD_ROLES = [(0, "next"), (6, "self"), (HEAD, "pair"), (5, "markers")]

P.show_head_gallery(run, LAYER, HEAD_ROLES, attention, pair=("she", "alice"))

That is why a layer has more than one head. One head has a single row of
weights to spend, so it can only afford one habit. Twelve side by side let
the layer watch position, keep a word intact, and tie a pronoun to a name all
at once. This afternoon's lecture starts from there.

---

# Act 4. What the square root is for

The slide carries a note: the scores, the `Q @ K.T` of task 1, are divided
by `sqrt(d)` "to avoid gradient vanishing with the softmax". Act 1 already showed the
mechanism. Bigger scores make a sharper softmax, and sharp enough scores turn
the softmax into a hard pick.

A query in this model has `d = 64` numbers. A dot product of 64 numbers adds
up 64 products, about as many negative as positive, so most of them cancel
and the total is typically only about 8 times the size of one product, and
`sqrt(64) = 8`.

First, let's test the claim made above that the total from d=64 products
grows like sqrt(d). The cell below draws thousands of random query/key pairs
at different sizes of d and measures how big their dot product turns out to be.

In [ ]:
rng = np.random.default_rng(0)

d_values = [8, 16, 32, 64, 128, 256, 512, 1024]
n_trials = 20000
measured_std = []

for d in d_values:
    q = rng.normal(size=(n_trials, d))
    k = rng.normal(size=(n_trials, d))
    measured_std.append((q * k).sum(axis=1).std())

P.show_score_growth(d_values, measured_std)

print("d       measured std   sqrt(d)")
for d, m in zip(d_values, measured_std):
    print("{:<8}{:<15.2f}{:.2f}".format(d, m, np.sqrt(d)))

A dot product of d random numbers adds up d products. Most of them cancel
and what survives grows with sqrt(d). Dividing by sqrt(d) undoes that growth.
Dividing by it brings the scores back to a size the softmax can work with.
Here is the same real head with and without the division:

In [ ]:
Q_r, K_r, V_r = project(run.hidden[LAYER], WQ_r, WK_r, WV_r, bQ_r, bK_r, bV_r)
P.show_scaling_effect(Q_r, K_r, run.tokens)

Without the division nearly every row becomes a hard pick. A hard pick is a
problem during training: when one weight is 1 and the rest are 0, a small
change in `WQ` or `WK` leaves the weights at 1 and 0, so training gets no
signal about where the token should look. That is the vanishing gradient the
slide warns about. The square root keeps the softmax soft enough to learn.

## 🎯 Bonus: does it always get it right?
    
Every example so far was chosen because it worked. Real models are not
perfect. Pick a sentence with a genuinely ambiguous pronoun, one a human
would also have to think about, and see what the model's heads do with it.
    
`it` in the sentence below could grammatically point to either noun. A
person resolves it instantly using real world reasoning about size and
fitting, not grammar. Does the model track that, or does it just favor one
noun regardless of what actually makes sense?

In [ ]:
SENTENCE_3 = "The trophy didn't fit in the suitcase because it was too big."
TOKEN, TARGET_A, TARGET_B = "it", "trophy", "suitcase"
    
run3 = R.Run(SENTENCE_3)
print("tokens:", run3.tokens)
print()
    
best_a = run3.best_heads_for(TOKEN, TARGET_A)[0]
best_b = run3.best_heads_for(TOKEN, TARGET_B)[0]
print("best head for 'it' -> '{}': layer {}, head {}, weight {:.3f}".format(TARGET_A, *best_a))
print("best head for 'it' -> '{}': layer {}, head {}, weight {:.3f}".format(TARGET_B, *best_b))
    
winner = TARGET_A if best_a[2] > best_b[2] else TARGET_B
print()
print("the strongest head favors '{}'".format(winner))

As you can see, the model is not that confident when it comes to ambiguous
cases since the difference in weights is very thin.
Now change `"too big"` to `"too small"` and rerun. Now `it` means the suitcase, not the trophy. 
Does the model's strongest head reflect that?
    
If it does: the model is tracking something like real world meaning, not
just grammar or word order. If it does not: the head learned a shortcut,
perhaps always favoring whichever noun sits closer to `it'.

---

## ⭐ Optional: your own sentence

For whoever has minutes left. The exercise is complete above.

A sentence from the office this time. Who is *it*? Change `SENTENCE_2`,
`TOKEN` and `TARGET` and ask the model. `best_heads_for` searches all 72
heads for the ones that send the most weight from one token to another, and
`where_does_it_look` lists, for one head, the words a token puts the most
weight on. Tokens are lowercase, and the model may split a rare word into
pieces marked with `##`; the printed token list shows what it did. A word
that occurs more than once can be given as its position in that list.

In [ ]:
SENTENCE_2 = "The bank approved the loan after the manager reviewed it carefully."
TOKEN, TARGET = "it", "loan"

run2 = R.Run(SENTENCE_2)
print("tokens:", run2.tokens)
print()

# Which heads send the most weight from TOKEN to TARGET? (layer, head, weight), best first.
for L, h, w in run2.best_heads_for(TOKEN, TARGET)[:3]:
    print("layer {} head {}: '{}' puts {:.0%} of its attention on '{}'".format(L, h, TOKEN, w, TARGET))

# Take the best one and look at the whole row.
L, h, _ = run2.best_heads_for(TOKEN, TARGET)[0]
print()
print("In layer {} head {}, '{}' looks at:".format(L, h, TOKEN))
for tok, w in run2.where_does_it_look(L, h, TOKEN)[:4]:
    print("   {:12s} {:.0%}".format(tok, w))

S2, _ = attention(run2.hidden[L], *run2.head_weights(L, h))
P.show_map(S2, run2.tokens, "Layer {} head {}: where '{}' looks".format(L, h, TOKEN))

---

    


# Takeaway

- **Attention is a soft lookup.** Score a query against every key, softmax
  the scores into weights, blend the values by those weights.
- **In a sentence every token is query, key and value at once.** Three
  matrices, `WQ`, `WK`, `WV`, decide how, and training chooses them.
- **The output is an average.** Each token leaves the head as a weighted
  average of the values, weighted by what it attended to.
- **Your five lines are the real thing.** They reproduced a production model's
  attention maps to six decimal places.

## What comes next

This afternoon the lecture stacks many heads side by side, tells the model
where each word sits in the sentence, and piles layers on top of each other.
That stack is a transformer. Every attention head inside it is the one you
wrote today.